# Q1. Supervised Learning — Heart Disease Classification

**Objective:** Build and evaluate classification models to predict whether a patient has heart disease.

**Target column:** `heart_disease`  
- `1` = disease present  
- `0` = disease absent

This notebook covers data loading, inspection, EDA, preprocessing, model training, evaluation, and hyperparameter tuning.


## 1. Data Loading and Inspection


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


In [ ]:
df = pd.read_csv("q1_heart_disease.csv")

print("Dataset Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

df.head()


### Inspection Summary

The dataset contains patient health-related features and the target variable `heart_disease`. Missing values are present in `resting_bp` and `cholesterol`, so they will be handled during preprocessing using imputation instead of deleting rows.


## 2. Exploratory Data Analysis


### Plot 1: Target Class Distribution


In [ ]:
class_counts = df["heart_disease"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Target Class Distribution")
plt.xlabel("Heart Disease (0 = Absent, 1 = Present)")
plt.ylabel("Number of Patients")
plt.show()

print(class_counts)
print("\nPercentage distribution:")
print(df["heart_disease"].value_counts(normalize=True).sort_index().mul(100).round(2))


**Interpretation:**  
This chart shows the number of patients in each target class. Since both classes are present in a comparable proportion, stratified train-test splitting is suitable because it preserves the class ratio in both training and testing data.


### Plot 2: Correlation Heatmap


In [ ]:
corr_df = pd.get_dummies(df, drop_first=True)
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 8))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Correlation Heatmap of Numerical and Encoded Features")
plt.tight_layout()
plt.show()


**Interpretation:**  
The heatmap shows relationships between features and the target variable. Variables with stronger positive or negative correlation with `heart_disease` may be more useful for prediction. It also helps identify whether any features are highly correlated with each other.


### Plot 3: Chest Pain Type vs Heart Disease


In [ ]:
cp_table = pd.crosstab(df["chest_pain_type"], df["heart_disease"])

cp_table.plot(kind="bar", figsize=(8, 5))
plt.title("Chest Pain Type vs Heart Disease")
plt.xlabel("Chest Pain Type")
plt.ylabel("Number of Patients")
plt.xticks(rotation=25)
plt.legend(title="Heart Disease")
plt.tight_layout()
plt.show()

cp_table


**Interpretation:**  
This chart compares heart disease outcomes across different chest pain types. Some chest pain categories show a higher number of disease cases, so `chest_pain_type` is likely to be an important categorical feature for the model.


### Plot 4: Age Distribution by Heart Disease Status


In [ ]:
age_no_disease = df.loc[df["heart_disease"] == 0, "age"]
age_disease = df.loc[df["heart_disease"] == 1, "age"]

plt.figure(figsize=(8, 5))
plt.boxplot([age_no_disease, age_disease], labels=["No Disease", "Disease"])
plt.title("Age Distribution by Heart Disease Status")
plt.xlabel("Heart Disease Status")
plt.ylabel("Age")
plt.show()

df.groupby("heart_disease")["age"].describe()


**Interpretation:**  
The boxplot compares age distribution for patients with and without heart disease. If the disease group has a different median age or spread, age may help the classifier separate the two classes.


## 3. Data Preprocessing


### Missing Value Strategy

The missing values are handled using **median imputation** for numerical columns and **most frequent imputation** for categorical columns.

Median imputation is chosen for numerical features because it is more robust to outliers than mean imputation. Dropping rows is avoided because it would reduce the dataset size and may remove useful patient records. The imputation is placed inside a scikit-learn pipeline to avoid data leakage.


In [ ]:
X = df.drop(columns=["heart_disease"])
y = df["heart_disease"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Numerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\nTraining set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(3))


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

preprocessor


## 4. Model Training


In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

trained_models = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    trained_models[model_name] = pipeline
    
    print(f"{model_name} trained successfully.")


## 5. Model Evaluation


In [ ]:
results = []

for model_name, pipeline in trained_models.items():
    y_pred = pipeline.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })
    
    print("=" * 60)
    print(model_name)
    print("=" * 60)
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

results_df = pd.DataFrame(results).sort_values(by="F1-score", ascending=False)
results_df


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print("Best model based on F1-score:", best_model_name)


### Best Model Conclusion

The best-performing model is selected using the test-set metrics, especially **F1-score**, because this problem requires balancing precision and recall. Precision tells us how many predicted disease cases were actually disease cases, while recall tells us how many actual disease cases were correctly detected.

In the prepared run, the **Random Forest** model gave the best balance of precision, recall, and F1-score among the three models. Therefore, Random Forest is selected for hyperparameter tuning.


## 6. Hyperparameter Tuning with GridSearchCV


In [ ]:
baseline_rf = trained_models["Random Forest"]
baseline_pred = baseline_rf.predict(X_test)

baseline_metrics = {
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred),
    "Recall": recall_score(y_test, baseline_pred),
    "F1-score": f1_score(y_test, baseline_pred)
}

baseline_metrics


In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters Found:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1-score:")
print(round(grid_search.best_score_, 4))


In [ ]:
tuned_model = grid_search.best_estimator_
tuned_pred = tuned_model.predict(X_test)

tuned_metrics = {
    "Accuracy": accuracy_score(y_test, tuned_pred),
    "Precision": precision_score(y_test, tuned_pred),
    "Recall": recall_score(y_test, tuned_pred),
    "F1-score": f1_score(y_test, tuned_pred)
}

comparison_df = pd.DataFrame(
    [baseline_metrics, tuned_metrics],
    index=["Untuned Random Forest", "Tuned Random Forest"]
)

print("Tuned Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, tuned_pred))

print("\nTuned Random Forest Classification Report:")
print(classification_report(y_test, tuned_pred, target_names=["No Disease", "Disease"]))

comparison_df


### Hyperparameter Tuning Conclusion

The tuned Random Forest model is compared with the untuned Random Forest baseline using accuracy, precision, recall, and F1-score. The final decision should be based mainly on F1-score because it balances false positives and false negatives.

If the tuned model improves or maintains the F1-score while keeping recall and precision strong, it is the better final model. If tuning gives only a small improvement, the baseline model is still acceptable, but the tuned version is preferred because its parameters were selected using cross-validation.


## Final Summary

This notebook completed the full supervised learning workflow:

1. Loaded and inspected the heart disease dataset.
2. Created EDA visualisations and interpreted each chart.
3. Handled missing values using imputation.
4. Encoded categorical variables and scaled numerical variables.
5. Trained Decision Tree, Random Forest, and Gradient Boosting classifiers.
6. Evaluated each model using confusion matrix, precision, recall, and F1-score.
7. Tuned the best model using GridSearchCV and compared tuned vs untuned performance.
